In [11]:
SOURCE_PATH = "/kaggle/input/datasets/obulisainaren/multi-cancer/Multi Cancer/Multi Cancer"

DESTINATION_PATH = "/kaggle/working/flattened_dataset"

In [12]:
import shutil
import os


if os.path.exists(DESTINATION_PATH):
    shutil.rmtree(DESTINATION_PATH)

print("Old flattened dataset deleted.")

Old flattened dataset deleted.


In [13]:
os.makedirs(DESTINATION_PATH, exist_ok=True)

print("New destination folder created.")

New destination folder created.


In [15]:
import shutil
import os
from tqdm import tqdm

total_copied = 0

for cancer_type in os.listdir(SOURCE_PATH):

    cancer_path = os.path.join(SOURCE_PATH, cancer_type)

    for subtype in os.listdir(cancer_path):

        subtype_path = os.path.join(cancer_path, subtype)

        destination_subtype = os.path.join(DESTINATION_PATH, subtype)

        os.makedirs(destination_subtype, exist_ok=True)

        images = os.listdir(subtype_path)

        for image_name in tqdm(images, desc=subtype, leave=False):

            source_image = os.path.join(subtype_path, image_name)

            destination_image = os.path.join(destination_subtype, image_name)

            shutil.copy2(source_image, destination_image)

            total_copied += 1

print(f"\n✅ Dataset Flattened Successfully!")
print(f"📸 Total Images Copied: {total_copied}")


✅ Dataset Flattened Successfully!
📸 Total Images Copied: 130002


In [48]:
print("Classes:", len(os.listdir(DESTINATION_PATH)))

print(
    "brain_glioma images:",
    len(os.listdir(os.path.join(DESTINATION_PATH, "brain_glioma")))
)

Classes: 26
brain_glioma images: 5000


In [50]:
import os
import shutil

import torch
import torch.nn as nn

from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

In [51]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [52]:
dataset = ImageFolder(
    root=DESTINATION_PATH,
    transform=transform
)

In [53]:
image, label = dataset[0]

print(type(image))
print(image.shape)
print(label)

<class 'torch.Tensor'>
torch.Size([3, 224, 224])
0


In [56]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset,
    batch_size=64,      # was 32
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [57]:
images, labels = next(iter(train_loader))

print(type(labels))
print(labels.shape)

<class 'torch.Tensor'>
torch.Size([64])


In [58]:
import torch
import torch.nn as nn

class CancerCNN(nn.Module):

    def __init__(self):
        super().__init__()

        # Feature Extraction
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2,2)

        # Flatten Layer
        self.flatten = nn.Flatten()

        # Fully Connected Layers
        self.fc1 = nn.Linear(32*56*56,128)
        self.fc2 = nn.Linear(128,26)

    def forward(self,x):

        # Block 1
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        # Block 2
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        # Flatten
        x = self.flatten(x)

        # Classifier
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x

In [59]:
# Get a fresh batch from the DataLoader
images, labels = next(iter(train_loader))

print(type(images))
print(images.shape)

print(type(labels))
print(labels.shape)

<class 'torch.Tensor'>
torch.Size([64, 3, 224, 224])
<class 'torch.Tensor'>
torch.Size([64])


In [60]:
model = CancerCNN()

output = model(images)

print("Input Shape :", images.shape)
print("Output Shape:", output.shape)

Input Shape : torch.Size([64, 3, 224, 224])
Output Shape: torch.Size([64, 26])


In [61]:
criterion = nn.CrossEntropyLoss()

output = model(images)

loss = criterion(output, labels)

print("Output Shape :", output.shape)
print("Labels Shape :", labels.shape)
print("Loss :", loss)

Output Shape : torch.Size([64, 26])
Labels Shape : torch.Size([64])
Loss : tensor(3.2564, grad_fn=<NllLossBackward0>)


In [63]:
from tqdm import tqdm
import time

num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    start = time.time()

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs}"
    )

    for images, labels in progress_bar:

        # Move batch to GPU
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        progress_bar.set_postfix(
            Loss=f"{loss.item():.4f}"
        )

    epoch_loss = running_loss / len(train_loader)

    print(
        f"\nEpoch [{epoch+1}/{num_epochs}] "
        f"Average Loss: {epoch_loss:.4f} "
        f"Time: {time.time()-start:.2f}s"
    )

Epoch 1/10: 100%|██████████| 2032/2032 [06:35<00:00,  5.14it/s, Loss=0.6032]



Epoch [1/10] Average Loss: 0.5876 Time: 395.02s


Epoch 2/10: 100%|██████████| 2032/2032 [06:30<00:00,  5.20it/s, Loss=0.2454]



Epoch [2/10] Average Loss: 0.2664 Time: 390.98s


Epoch 3/10: 100%|██████████| 2032/2032 [06:30<00:00,  5.21it/s, Loss=0.0308]


...


Epoch [8/10] Average Loss: 0.0432 Time: 391.22s


Epoch 9/10: 100%|██████████| 2032/2032 [06:37<00:00,  5.11it/s, Loss=0.0011]



Epoch [9/10] Average Loss: 0.0436 Time: 397.44s


Epoch 10/10: 100%|██████████| 2032/2032 [06:30<00:00,  5.20it/s, Loss=0.0001]


Epoch [10/10] Average Loss: 0.0364 Time: 390.95s


In [66]:
torch.save(model.state_dict(), "cancer_cnn.pth")

print("Model saved successfully!")


Model saved successfully!
